First all the imports at the top, if you need new imports add here

In [8]:
import sys
import time
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import uuid

The code below open the driver and opens the results page of the official hyrox site

In [28]:
driver = webdriver.Chrome()
driver.get("https://results.hyrox.com/season-8/")
time.sleep(2)
soup = BeautifulSoup(driver.page_source, "html.parser")

This next block selects the filters (testing things like amsterdam, men, 100 results for now), and then clicks on show results button to actually get to the data.

In [ ]:
wait = WebDriverWait(driver, 10)

# This is where we could implement a for loop to select the different cities etc.
selected_city = "2026 Amsterdam"
selected_hyrox_type = "HYROX - Overall"
selected_gender = "M"

# Select the city
city_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-event_main_group")
    ))
)
# get a list of all the cities
cities = [
    option.get_attribute("value")
    for option in city_dropdown.options
]
#To be changed later to check different cities, does nothing yet and may not be right
cities_to_check = ["2026 Amsterdam", "2026 Rotterdam", "2026 Heerenveen", "2026 Mechelen", "2026 Gent", "2026 Maastricht", "2026 utrecht"]

#actually click amsterdam
city_dropdown.select_by_value(selected_city)

#allow page to update, the other dropdowns get reloaded after picking a city
time.sleep(2)

# We want only the overall hyrox so get the different types of events
# have to find it again here because the page changed after picking the city
event_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-event")
    ))
)
# Create a list of the different types, may be useful later
event_types = [
    option.get_attribute("value")
    for option in event_dropdown.options
]
# use the visible label HYROX overall label and select it
event_dropdown.select_by_visible_text(selected_hyrox_type)

# The same code but this time for the gender filter
gender_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-sex")
    ))
)
# Choose just men for now to check if everything works
gender_dropdown.select_by_value(selected_gender)

# Same code but for the number of rows per page
results_per_page = Select(
    wait.until(EC.presence_of_element_located((By.ID, "default-num_results")))
)
#Select 100 results per page to make scraping faster
results_per_page.select_by_value("100")

#give the page a moment to settle after all the filter changes
time.sleep(2)

#wait until the show results button can actually be clicked --> earlier the button sometimes wouldn't click
submit_button = wait.until(EC.element_to_be_clickable((By.ID, "default-submit")))

#click the show results button which is named default submit in the inspect tool
driver.find_element(By.ID, "default-submit").click()

Before you continue, check whether the driver worked correctly. It should now be on the results page!

Then the code below is the next step, it extracts the data we need from this page using beautifulsoup, then clicks next page and does the same until there is no more next page (meaning all the rows for that event with its specific filters are extracted)

In [ ]:
import uuid
from urllib.parse import urljoin

soup = BeautifulSoup(driver.page_source, "html.parser")

rows = soup.select(
    'div[data-sex="M"] ul.list-group > li.list-group-item:not(.list-group-header)'
)
print("Result rows found:", len(rows))

collection_id = str(uuid.uuid4())
results_ranking_page = []

for row in rows:
    flag = row.find("img", class_="nation__icon")
    abbreviation = flag.get("alt") if flag else None

    link_tag = row.find("a", href=True)
    personal_link = urljoin(driver.current_url, link_tag["href"]) if link_tag else None #we need the full link instead of the relative link otherwise it did not woerk

    age_field = row.find("div", class_="type-age_class")
    age_group = age_field.get_text(" ", strip=True).replace("Age Group", "").strip() if age_field else None

    time_field = row.find("div", class_="type-time")
    total_time = time_field.get_text(" ", strip=True).replace("Total", "").strip() if time_field else None

    results_ranking_page.append({
        "collection_id": collection_id,
        "record_id": str(uuid.uuid4()),
        "nationality_athlete": abbreviation,
        "city_of_event": selected_city
        "age_group": age_group,
        "total_time": total_time,
        "personal_link": personal_link
    })

print(results_ranking_page[:5])


Result rows found: 100
[{'collection_id': 'defc2568-55a1-47c4-af3e-1507ee0a7248', 'record_id': '00509c6a-fa55-4f6e-ae4f-aed26be4e2b8', 'country': 'GER', 'age_group': '35-39', 'total_time': '00:53:43', 'personal_link': 'https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI45806E_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL'}, {'collection_id': 'defc2568-55a1-47c4-af3e-1507ee0a7248', 'record_id': '8c12d256-bbaa-4522-a783-0c1a5a277d01', 'country': 'NED', 'age_group': '40-44', 'total_time': '00:54:55', 'personal_link': 'https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI456F3F_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL'}, {'collection_id'